## Import all what we need.

In [64]:
# # encoding: utf-8
# import os
# import bz2
# import glob
# import time
# import requests
# import argparse
# import subprocess
# import numpy as np
# import pandas as pd
# import numexpr as ne
# import dask.array as da
# import astropy.units as au
# import dask.dataframe as dd
# import astropy.constants as ac
# import matplotlib.pyplot as plt
# from io import StringIO
# from itertools import chain
# from tqdm.notebook import tqdm
# from pandarallel import pandarallel
# from matplotlib.collections import LineCollection
# from concurrent.futures import ThreadPoolExecutor
# from multiprocessing import Process, Pool, freeze_support
# from matplotlib.ticker import MultipleLocator, FormatStrFormatter
# from scipy.special import voigt_profile, wofz, erf, roots_hermite

# import warnings
# warnings.simplefilter("ignore", np.ComplexWarning)
# pd.options.mode.chained_assignment = None

# import matplotlib as mpl
# mpl.rcParams['agg.path.chunksize'] = 10000

# import multiprocessing as mp
# freeze_support()
# num_cpus = mp.cpu_count()
# print('Number of CPU: ', num_cpus)

# # import vaex

In [65]:
# Import all what we need.
import time
import requests
import argparse
import numpy as np
import pandas as pd
import numexpr as ne
from io import StringIO

### TODO Remove me ###
import sys, os
# webapp_path = '/Users/christian/www/ExoMolHR-web/exomolhr'
webapp_path = '/home/jingxin/ExoMolHR-web/exomolhr'
sys.path.append(webapp_path)
os.environ['DJANGO_SETTINGS_MODULE'] = 'exomolhr.settings'
# Prepare the Django models
import django
django.setup()
### TODO Remove me ###

from django.conf import settings

# Check the number of this computer's CPUs.
import multiprocessing
cup_num = multiprocessing.cpu_count()
print(f"CPU number: {cup_num}")
from pandarallel import pandarallel
pandarallel.initialize(nb_workers=4,progress_bar=False)    # Initialize.
#pandarallel.initialize(nb_workers=16,progress_bar=True)    # Initialize.

CPU number: 256
INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [66]:
# # Path and Parameters.
# '''Could be changed !'''
# #########################################################
# unc_path = settings.DATA_DIR / 'uncertainty_list.csv'
# database_path = settings.EXOMOL_DATA_DIR
# loc_result_path = settings.LOCAL_CSV_DIR / 'loc_result'
# web_result_path = settings.EXOMOLHR_RESULTS_DIR

# T = 296
# iso = '12C-16O2'
# min_frequency = 600.00
# max_frequency = 800.00
# max_uncertainty = 0.01 * 100
# min_intensity = 1E-30
# select_qn_label = ['Gtot', 'e/f', 'n1', 'n2', 'l2', 'n3']

# #########################################################


# from scipy.constants import h, c, k as kB
# c *= 100 
# c2 = h * c / kB                   # Second radiation constant (cm K)
# _c2_T = - c2 / T
# pi_c_8 = 1 / (8 * np.pi * c)      # 8 * pi * c (cm-1 s)

In [67]:
# Path and Parameters.
'''Could be changed !'''
#########################################################
unc_path = settings.DATA_DIR / 'uncertainty_list.csv'
database_path = settings.EXOMOL_DATA_DIR
loc_result_path = settings.LOCAL_CSV_DIR / 'loc_result'
web_result_path = settings.EXOMOLHR_RESULTS_DIR

T = 296
iso = ['12C-16O2', '27Al-16O']
min_frequency = 600.00
max_frequency = 800.00
max_uncertainty = 0.01 * 100
min_intensity = 1E-30
select_qn_label = ['Gtot', 'e/f', 'n1', 'n2', 'l2', 'n3', 'ElecState']

#########################################################


from scipy.constants import h, c, k as kB
c *= 100 
c2 = h * c / kB                   # Second radiation constant (cm K)
_c2_T = - c2 / T
pi_c_8 = 1 / (8 * np.pi * c)      # 8 * pi * c (cm-1 s)

## Report time

In [68]:
class Timer:    
    def start(self):
        self.start_CPU = time.process_time()
        self.start_sys = time.time()
        return self

    def end(self, *args):
        self.end_CPU = time.process_time()
        self.end_sys = time.time()
        self.interval_CPU = self.end_CPU - self.start_CPU
        self.interval_sys = self.end_sys - self.start_sys
        print('{:25s} : {}'.format('Running time on CPU', self.interval_CPU), 's')
        print('{:25s} : {}'.format('Running time on system', self.interval_sys), 's')


In [69]:
# Part 1: Get Molecule, Isotopologue, Dataset and Abundance.
'''
Get the names of molecule name, isotopologue name and dataset name from the api__urls.txt 
which saved the URLs with molecule, isotopologue and dataset. 
Combine them with '/' for reading files from folders more convenient later.
'''
def mol_param(isotopologue):
    colnames=['id','molecule','isotopologue','isoformula','dataset','abundance','Main column', 'Main format', 'QN label','QN format']
    molparam_df = pd.read_csv(unc_path, usecols=[0,1,2,3,4,5,6,7,8,9], names=colnames, header=0)     
    molparam = molparam_df[molparam_df['isotopologue'].isin([isotopologue])]
    molecule = molparam['molecule'].values[0]
    isoformula = molparam['isoformula'].values[0]
    dataset = molparam['dataset'].values[0]
    abundance = molparam['abundance'].values[0]
    
    J_format = str(molparam['Main format'].values).split(',')[3]
    qns_label = molparam['QN label'].values[0]
    qns_format = molparam['QN format'].values[0]
    mol_iso_ds_path = molecule + '/' + isotopologue + '/' + dataset
    print('Molecule \t\t:', molecule)
    print('Isotopologue \t:', isotopologue) 
    print('Isotopologue formula \t\t:', isoformula) 
    print('Dataset \t\t:', dataset)  
    print('Abundance \t\t:', abundance) 
    print('J format \t\t:', J_format)
    print('Quantumn number labels \t\t:', qns_label)
    print('Quantumn number formats \t:', qns_format, '\n')    
    return (molparam_df, molecule, isoformula, dataset, abundance, 
            J_format, qns_label, qns_format, mol_iso_ds_path)


Molecule 		: CO2
Isotopologue 	: 12C-16O2
Isotopologue formula 		: (12C)(16O)2
Dataset 		: UCL-4000
Abundance 		: 1
J format 		: %7d
Quantumn number labels 		: Gtot,e/f,n1,n2,l2,n3,m1,m2,m3,m4,m5
Quantumn number formats 	: %3s,%1s,%2d,%2d,%2d,%2d,%2d,%2d,%2d,%2d,%2d 

Read the local result file.
['Gtot', 'e/f', 'n1', 'n2', 'l2', 'n3', 'm1', 'm2', 'm3', 'm4', 'm5']
Read the partition function file.
The partition function at T = 296 K is 286.0983 

Molecule 		: AlO
Isotopologue 	: 27Al-16O
Isotopologue formula 		: (27Al)(16O)
Dataset 		: ATP
Abundance 		: 1
J format 		: %7.1f
Quantumn number labels 		: +/-,e/f,ElecState,v,Lambda,Sigma,Omega
Quantumn number formats 	: %1s,%1s,%12s,%3d,%3d,%5.1f,%5.1f 

Read the local result file.
['+/-', 'e/f', 'ElecState', 'v', 'Lambda', 'Sigma', 'Omega']
Read the partition function file.
The partition function at T = 296 K is 3911.9208 



In [106]:
result_df

,Frequency,Uncertainty,A,Intensity,Molecule,Isotopologue,Dataset,"E""",g',"g""",J',"J""",QN',"QN"""
5453,600.105272,0.005025,0.12003,5.747903e-28,CO2,12C-16O2,UCL-4000,3740.879703,31,29,15.0,14.0,A1 f 0 3 1 1,A2 f 0 2 2 1
5462,600.271522,0.005025,0.62778,6.685989e-29,CO2,12C-16O2,UCL-4000,4895.665693,189,191,94.0,95.0,A2 f 0 3 3 0,A1 f 0 2 2 0
5468,600.358095,0.005025,0.45950,3.143985e-28,CO2,12C-16O2,UCL-4000,4521.458604,197,199,98.0,99.0,A1 e 0 2 2 0,A2 e 0 1 1 0
5480,600.633481,0.005025,0.12134,5.835349e-28,CO2,12C-16O2,UCL-4000,3752.537025,33,31,16.0,15.0,A1 e 0 3 1 1,A2 e 0 2 2 1
5485,600.770100,0.005001,0.45065,1.836537e-25,CO2,12C-16O2,UCL-4000,3186.954100,179,181,89.0,90.0,A2 e 0 1 1 0,A1 e 0 0 0 0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5379,744.314175,0.006058,3.46960,1.246116e-30,AlO,27Al-16O,ATP,5549.102909,312,300,25.5,24.5,e A(2PI),e A(2PI)
5418,745.096874,0.005156,3.48600,1.137511e-30,AlO,27Al-16O,ATP,5576.186731,324,312,26.5,25.5,e A(2PI),e A(2PI)
5419,745.103239,0.004795,3.47370,1.133513e-30,AlO,27Al-16O,ATP,5576.180553,324,312,26.5,25.5,f A(2PI),f A(2PI)
5443,745.878852,0.005473,3.48880,1.027669e-30,AlO,27Al-16O,ATP,5604.315455,336,324,27.5,26.5,f A(2PI),f A(2PI)


In [70]:
## 2.1 Read Partition Function File.
# Read partition function with online webpage.
def read_exomol_pf(pf_filename, pf_col_name):
    print('Read the partition function file.')
    pf_df = pd.read_csv(pf_filename, sep='\\s+', names=pf_col_name, header=None)
    return pf_df

# Read partition function with local partition function file.
def read_web_pf(pf_filename, pf_col_name):
    pf_url = pf_filename.replace(str(database_path), 'https://exomol.com/db')
    response = requests.get(pf_url)
    if response.status_code == 200:
        print('Partition function file webpage exists.')
    else:
        raise Exception('Partition function file webpage does not exist.')   
    pf_content = response.text
    print('Read the partition function file.')
    pf_df = pd.read_csv(StringIO(pf_content), sep=r'\s+', names=pf_col_name, header=None) 
    return pf_df

def get_Q(T, mol_iso_ds_path, isotopologue, dataset):
    pf_filename = str(database_path / (mol_iso_ds_path + '/' + isotopologue + '__' + dataset + '.pf'))
    pf_col_name = ['T', 'Q']
    if os.path.exists(pf_filename):
        pf_df = read_exomol_pf(pf_filename, pf_col_name)
    else:
        pf_df = read_web_pf(pf_filename, pf_col_name)

    max_T = pf_df.count()[0]
    inNumberint = int(T)
    if T != inNumberint:
        raise Exception('Sorry, please type an integer as temperature.')
    elif T > max_T:
        print('The maximum temperature is', max_T, 'K.')
        raise Exception('Sorry, please type a smaller T.')
    elif T < 1:
        raise('Sorry, please type a new T which is larger than or equal to 1')
    else:
        Q = pf_df['Q'][T-1]
        print('The partition function at T =', T, 'K is', Q, '\n')
    return(Q)


In [71]:
## 2.2 Calculating.

# Calculate intensity at the chosen temperature.
def cal_intensity(A, Epp, gp, Q, nu, abundance):
    I = ne.evaluate('gp * A * exp(_c2_T * Epp) * (1 - exp(_c2_T * nu)) * pi_c_8 / (nu ** 2) / Q * abundance')
    return I

In [115]:
## 2.3 Format the results.

### 2.3.1 Format the Quantumn Numbers
def format_qns(loc_df, qns_label, qns_format):
    qn_label_list = qns_label.split(',')
    qn_format_list = qns_format.split(',')
    print(qn_label_list)
    select_qn_label_index = [i for i in range(len(qn_label_list)) if qn_label_list[i] in select_qn_label]
    select_qn_lab = [qn_label_list[i] for i in select_qn_label_index]
    select_qn_fmt = [qn_format_list[i] for i in select_qn_label_index]
    select_qn_format = [format.replace(format[1:-1],str(pd.to_numeric(format[1:-1])+1)).replace("%",'{: >')+'}' for format in select_qn_fmt]
    num_select_qn = len(select_qn_format)
    qn_label_u_list = [loc_df[select_qn_lab[i]+"'"].map(select_qn_format[i].format) 
                    for i in range(num_select_qn)]
    qn_label_l_list = [loc_df[select_qn_lab[i]+'"'].map(select_qn_format[i].format) 
                    for i in range(num_select_qn)]
    max_qn_format_num = sum([int(char) for item in select_qn_format for char in item if char.isdigit()])+num_select_qn-1
    qn_label_u = pd.DataFrame(qn_label_u_list).sum(axis=0)
    qn_label_l = pd.DataFrame(qn_label_l_list).sum(axis=0)
    return(max_qn_format_num, qn_label_u, qn_label_l)

In [74]:
def calc_results(loc_df, qn_label_u, qn_label_l, molecule, isotopologue, dataset, abundance):
    A = pd.to_numeric(loc_df['A']).values
    Epp = pd.to_numeric(loc_df['E"']).values
    gp = pd.to_numeric(loc_df["g'"]).values
    nu = pd.to_numeric(loc_df['Frequency']).values
    Q = get_Q(T, mol_iso_ds_path, isotopologue, dataset)
    I = cal_intensity(A, Epp, gp, Q, nu, abundance)
    web_df = pd.DataFrame()
    web_df[['Frequency', 'Uncertainty', 'A', 'E"', "g'", 'g"', "J'", 'J"']] = loc_df[['Frequency', 'Uncertainty', 'A', 'E"', "g'", 'g"', "J'", 'J"']]
    web_df['Intensity'] = I
    web_df['Molecule'] = molecule
    web_df['Isotopologue'] = isotopologue
    web_df['Dataset'] = dataset
    web_df["QN'"] = qn_label_u
    web_df['QN"'] = qn_label_l
    web_df = web_df[web_df['Intensity'] > min_intensity]
    order = ['Frequency', 'Uncertainty', 'A', 'Intensity', 'Molecule', 'Isotopologue', 'Dataset', 'E"', "g'", 'g"', "J'", 'J"', "QN'", 'QN"']
    if web_df.empty:
        return web_df[order]
    web_df = web_df[order]
    return(web_df)

In [116]:
def read_loc_get_result(loc_result_path, molecule, isotopologue, dataset, abundance, mol_iso_ds_path, isoformula, qns_label, qns_format):   
    web_df = pd.DataFrame()
    loc_result_filepath = loc_result_path / (mol_iso_ds_path.replace('/','__') + '__' + isoformula  + '.csv')
    print('Read the local result file.')
    read_loc = pd.read_csv(loc_result_filepath, header=0, chunksize=1_000_000,
                            iterator=True, low_memory=False)
    for chunk in read_loc:
        chunk = chunk[pd.to_numeric(chunk['Frequency']).between(min_frequency, max_frequency)]
        chunk = chunk[pd.to_numeric(chunk['Uncertainty']) < max_uncertainty]
        if chunk.empty:
            continue
        (max_qn_format_num, qn_label_u, qn_label_l) = format_qns(chunk, qns_label, qns_format)
        web_format_chunk = calc_results(chunk, qn_label_u, qn_label_l, molecule, isotopologue, dataset, abundance)
        web_df = pd.concat([web_df, web_format_chunk])    
        
        
    # web_df = pd.DataFrame()
    # loc_result_filepath = loc_result_path / (mol_iso_ds_path.replace('/','__') + '__' + isoformula  + '.csv')
    # print('Read the local result file.')
    # read_loc = pd.read_csv(loc_result_filepath, header=0, chunksize=1_000_000,
    #                         iterator=True, low_memory=False)
    # for chunk in read_loc:
    #     chunk = chunk[pd.to_numeric(chunk['Frequency']).between(min_frequency, max_frequency)]
    #     chunk = chunk[pd.to_numeric(chunk['Uncertainty']) < max_uncertainty]
    #     if chunk.empty:
    #         continue
    #     (max_qn_format, qn_label_u, qn_label_l) = format_qns(chunk, qns_label, qns_format)
    #     web_format_chunk = calc_results(chunk, qn_label_u, qn_label_l, molecule, isot, dataset, abundance)
    #     web_df = pd.concat([web_df, web_format_chunk])    
    return(web_df, max_qn_format_num)



In [117]:
result_df = pd.DataFrame()
max_qn_fmt_list = []
for isot in iso:
    (molparam_df, molecule, isoformula, dataset, abundance, 
     J_format, qns_label, qns_format, mol_iso_ds_path) = mol_param(isot)
    (web_df, max_qn_format_num) = read_loc_get_result(loc_result_path, molecule, isot, dataset, abundance, 
                                                  mol_iso_ds_path, isoformula, qns_label, qns_format)  
    result_df = pd.concat([result_df, web_df])
    max_qn_fmt_list.append(max_qn_format_num)
max_qn_fmt = max(max_qn_fmt_list)
max_qn_format = '%'+str(max_qn_fmt)+'s'
save_folder = str(web_result_path)
if os.path.exists(save_folder):
    pass
else:
    os.makedirs(save_folder, exist_ok=True)
save_path = save_folder + '/' + '__'.join(iso) + '_web.hr'
result_df.sort_values(by=['Frequency'], ascending=True, inplace=True) 
fmt = '%12.6f%12.6f %10.4E %10.4E%10s%15s%10s %12.6f%6d%6d%7s%7s'+max_qn_format+max_qn_format
np.savetxt(save_path, result_df, fmt=fmt, delimiter=" ", header='')

Molecule 		: CO2
Isotopologue 	: 12C-16O2
Isotopologue formula 		: (12C)(16O)2
Dataset 		: UCL-4000
Abundance 		: 1
J format 		: %7d
Quantumn number labels 		: Gtot,e/f,n1,n2,l2,n3,m1,m2,m3,m4,m5
Quantumn number formats 	: %3s,%1s,%2d,%2d,%2d,%2d,%2d,%2d,%2d,%2d,%2d 

Read the local result file.
['Gtot', 'e/f', 'n1', 'n2', 'l2', 'n3', 'm1', 'm2', 'm3', 'm4', 'm5']
Read the partition function file.
The partition function at T = 296 K is 286.0983 

Molecule 		: AlO
Isotopologue 	: 27Al-16O
Isotopologue formula 		: (27Al)(16O)
Dataset 		: ATP
Abundance 		: 1
J format 		: %7.1f
Quantumn number labels 		: +/-,e/f,ElecState,v,Lambda,Sigma,Omega
Quantumn number formats 	: %1s,%1s,%12s,%3d,%3d,%5.1f,%5.1f 

Read the local result file.
['+/-', 'e/f', 'ElecState', 'v', 'Lambda', 'Sigma', 'Omega']
Read the partition function file.
The partition function at T = 296 K is 3911.9208 



In [130]:
result_df

,Frequency,Uncertainty,A,Intensity,Molecule,Isotopologue,Dataset,"E""",g',"g""",J',"J""",QN',"QN"""
5453,600.105272,0.005025,0.120030,5.747903e-28,CO2,12C-16O2,UCL-4000,3740.879703,31,29,15.0,14.0,A1 f 0 3 1 1,A2 f 0 2 2 1
5462,600.271522,0.005025,0.627780,6.685989e-29,CO2,12C-16O2,UCL-4000,4895.665693,189,191,94.0,95.0,A2 f 0 3 3 0,A1 f 0 2 2 0
5468,600.358095,0.005025,0.459500,3.143985e-28,CO2,12C-16O2,UCL-4000,4521.458604,197,199,98.0,99.0,A1 e 0 2 2 0,A2 e 0 1 1 0
5480,600.633481,0.005025,0.121340,5.835349e-28,CO2,12C-16O2,UCL-4000,3752.537025,33,31,16.0,15.0,A1 e 0 3 1 1,A2 e 0 2 2 1
5485,600.770100,0.005001,0.450650,1.836537e-25,CO2,12C-16O2,UCL-4000,3186.954100,179,181,89.0,90.0,A2 e 0 1 1 0,A1 e 0 0 0 0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104022,799.842435,0.007071,0.518860,5.855836e-30,CO2,12C-16O2,UCL-4000,4999.911601,57,57,28.0,28.0,A1 e 1 6 6 0,A2 f 0 7 7 0
104075,799.897105,0.007071,0.516470,6.261797e-30,CO2,12C-16O2,UCL-4000,4977.796693,55,55,27.0,27.0,A1 f 1 6 6 0,A2 e 0 7 7 0
104095,799.920401,0.007071,0.124640,4.391933e-30,CO2,12C-16O2,UCL-4000,4640.339008,31,29,15.0,14.0,A1 f 2 4 4 0,A2 f 0 7 5 0
104123,799.946994,0.007071,0.012651,4.212015e-26,CO2,12C-16O2,UCL-4000,2453.924501,71,73,35.0,36.0,A1 f 1 2 2 0,A2 f 0 3 1 0
